In [0]:
!pip install -U pypdf
!pip install -U langchain-text-splitters
!pip install -U databricks_langchain
!pip install pandas


In [0]:
dbutils.library.restartPython()

In [0]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [0]:
vol_landing_path="/Volumes/rag_on_databricks/landing/vol_landing/pdfs/"
dbutils.fs.ls(vol_landing_path)

In [0]:

pages=[]

for file in dbutils.fs.ls(vol_landing_path):
    reader=PdfReader(file.path.replace("dbfs:", ""))
    for page_num, page in enumerate(reader.pages,start=1):
        text= page.extract_text()
        pages.append(
            {
                "text":text,
                "page_num":page_num
            }
        )
print(len(pages)) #1219
    

In [0]:
pages[0]

In [0]:
%skip
from langchain_community.document_loaders import PyPDFLoader

# Initialize the loader with the path to your PDF file
loader = PyPDFLoader("path/to/your/document.pdf")

# Load the pages (each page becomes a LangChain Document object)
pages = loader.load()

# You can access the content and metadata of a specific page
print(pages[0].page_content)  # The text content of the page
print(pages[0].metadata)      # Metadata (e.g., source file path, page number)


In [0]:
# need to look  chunk size..
#Agentic RAG
# preprocessing and data cleaning..

In [0]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1500,chunk_overlap=300, separators=["\n\n", "\n", " ", ", ",""])
splitter.split_text(pages[0]["text"]) 

In [0]:
chunks=[]
for i, page in enumerate(pages):
    text=page["text"]
    chunks_subset=splitter.split_text(text)
    for j,chunk in enumerate(chunks_subset):
        chunks.append({
            "chunk":chunk,
            "id": f'chunk_id_{page["page_num"]}_{j}'
        })

print(len(chunks))

In [0]:
chunks[100]

In [0]:
import pandas as pd
import numpy as np

data = pd.DataFrame(chunks) #Chunks to convert that into dataframe and that df is added in delta table..
data.head()

In [0]:
from databricks_langchain import DatabricksEmbeddings

embedding_model = DatabricksEmbeddings(endpoint="databricks-bge-large-en")

In [0]:
np.array(embedding_model.embed_query("What is the meaning of life ?")).shape

In [0]:
# Generate embeddings for all chunks
import time

chunk_list = data["chunk"].tolist()
batch_size = 50  # Reasonable batch size for Databricks
all_embeddings = []

print(f"Processing {len(chunk_list)} chunks (batch_size={batch_size})\n")
start_time = time.time()

for i in range(0, len(chunk_list), batch_size):
    batch = chunk_list[i:i + batch_size]
    batch_embeddings = embedding_model.embed_documents(batch)
    all_embeddings.extend(batch_embeddings)
    print(f"Processed {min(i + batch_size, len(chunk_list))}/{len(chunk_list)} chunks")
    time.sleep(2)  # 2 second delay between batches to avoid rate limits

embeddings = all_embeddings
elapsed = time.time() - start_time
print(f"\n✅ Generated {len(embeddings)} embeddings in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Dimensions: {len(embeddings[0])}")

In [0]:
np.array(embeddings).shape #embedding to convert that into dataframe and that(data) df is added in delta table..


In [0]:
def normalize(vector):
    norms=np.linalg.norm(vector,axis=1,keepdims=True)
    norms[norms==0] =1e-12

    normalized_embds = (vector/norms) #make them unit vectors
    return normalized_embds



In [0]:
normalized_chunk= normalize(embeddings)


In [0]:
normalized_chunk.shape

In [0]:
def retrieve(query,k=3):
    query_vector = np.array(embedding_model.embed_query(query)).reshape(1,-1)
    
    normalized_query = normalize(query_vector)

    #It calculates similarity for every chunk.
    scores = (normalized_chunk @ normalized_query.T).flatten() #similarity scores 
    top_indices = np.argsort(scores)[::-1][:k]
    
    top_indices_results=[]
    for i in top_indices:
        top_indices_results.append(data.iloc[i].to_dict())
    
    return top_indices_results

#-------------------------------------------------------
# query_transposed = np.transpose(normalized_query)
# similarity_matrix = np.matmul(normalized_chunk, query_transposed)
# scores = similarity_matrix.flatten()

In [0]:
from databricks_langchain import ChatDatabricks

model = ChatDatabricks(
    endpoint="databricks-meta-llama-3-1-8b-instruct",
    max_tokens=500,
    temperature=0.1
)

In [0]:
def create_prompt(retrieved_sources, question):
    context = "\n\n".join(
        [
            f"Source {i+1}\n{doc['chunk']}"
            for i, doc in enumerate(retrieved_sources)
        ]
    )

    prompt = f"""
You are an expert AI assistant.

Use ONLY the information provided in the context below to answer the user's question.

Rules:
- Answer only from the provided context.
- Do not make up information.
- If the answer is not available in the context, reply:
  "I couldn't find the answer in the provided documents."
- Keep the answer clear, concise, and professional.
- Use bullet points whenever appropriate.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [0]:

def RAG(query):
    retrieved_sources = retrieve(query, k=4)

    prompt = create_prompt(retrieved_sources, query)

    response = model.invoke(prompt)

    answer=response.content
    return {
        "question": query,
        "answer": answer,
        "sources": retrieved_sources
    }

user_query = "what is the big data in the hadoop ecosystem ?" 
#"what is the capital of france ?"
#"What is sql language related with hadoop and data engineering context ?"
#"What is hdfs in hadoop ?"
#"What is mapreduce in the hadoop ecosystem ?"

RAG(user_query)


In [0]:
prompt

In [0]:
retrieved_sources